# Soccer Match Analysis
---
Comprehensive analysis of a football match using StatsBomb event data.

**Match:** Argentina vs France — FIFA World Cup 2022 Final
**Date:** 2022-12-18
**Result:** Argentina 3-3 France (Argentina won 4-2 on penalties)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from statsbombpy import sb
from mplsoccer import Pitch, VerticalPitch, FontManager
from scipy.ndimage import gaussian_filter

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('All imports loaded successfully.')

In [ ]:
# Load match data
MATCH_ID = 3869685  # Argentina vs France, 2022 World Cup Final

df = sb.events(match_id=MATCH_ID)
print(f'Total events: {len(df)}')
print(f'Columns: {len(df.columns.tolist())}')
df.head(3)

In [ ]:
# Match overview - lineups
lineups = sb.lineups(match_id=MATCH_ID)

print('LINEUPS:')
print('=' * 60)
for team, lineup in lineups.items():
    formation = lineup['formation'].iloc[0] if 'formation' in lineup.columns else 'N/A'
    print(f'\n{team} (Formation: {formation}):')
    print('-' * 40)
    for _, player in lineup.iterrows():
        pos = player.get('position_name', player.get('position_id', ''))
        jersey = player.get('jersey_number', '')
        print(f'  #{jersey} {player["player_name"]} -- {pos}')

print()
print('MATCH INFO')
print('=' * 60)
matches = sb.matches(competition_id=43, season_id=106)
match_info = matches[matches['match_id'] == MATCH_ID].iloc[0]
print(f'{match_info["home_team"]} {match_info["home_score"]} - {match_info["away_score"]} {match_info["away_team"]}')
print(f'Stadium: {match_info["stadium"]}')
print(f'Referee: {match_info["referee"]}')
print(f'Stage: {match_info["competition_stage"]}')

for col in ['home_manager_name', 'away_manager_name']:
    val = match_info.get(col, 'N/A')
    print(f'Manager ({col.split("_")[0]}): {match_info.get(col, "N/A")}')

In [ ]:
# Basic Statistics
teams = df['team'].unique()
print(f'Teams: {teams}')

stats = {}
for team in teams:
    team_events = df[df['team'] == team]
    t = {}
    t['Shots'] = len(team_events[team_events['type'] == 'Shot'])
    goals = team_events[(team_events['type'] == 'Shot') & (team_events['shot_outcome'] == 'Goal')]
    t['Goals'] = len(goals)
    sot = team_events[(team_events['type'] == 'Shot') & 
                      (team_events['shot_outcome'].isin(['Goal', 'Saved', 'Saved to Post']))]
    t['Shots on Target'] = len(sot)
    t['Passes'] = len(team_events[team_events['type'] == 'Pass'])
    passes = team_events[team_events['type'] == 'Pass']
    successful_passes = passes[passes['pass_outcome'].isna()]
    t['Pass Accuracy'] = f'{round(len(successful_passes) / len(passes) * 100, 1) if len(passes) > 0 else 0}%'
    t['Fouls'] = len(team_events[team_events['type'] == 'Foul Committed'])
    t['Offsides'] = len(team_events[team_events['type'] == 'Offside'])
    
    cards = team_events[team_events['type'] == 'Card']
    t['Yellow'] = len(cards[(cards['bad_behaviour_card'] == 'Yellow Card') | (cards['foul_committed_card'] == 'Yellow Card')])
    t['Red'] = len(cards[(cards['bad_behaviour_card'] == 'Red Card') | (cards['foul_committed_card'] == 'Red Card')])
    
    t['Tackles'] = len(team_events[team_events['type'] == 'Duel'])
    t['Interceptions'] = len(team_events[team_events['type'] == 'Interception'])
    t['Clearances'] = len(team_events[team_events['type'] == 'Clearance'])
    t['Dribbles'] = len(team_events[team_events['type'] == 'Dribble'])
    stats[team] = t

stats_df = pd.DataFrame(stats).T
stats_df.index.name = 'Team'
print('\n===== BASIC STATISTICS =====')
stats_df

In [ ]:
# Possession & Passing Analysis
print('Available pass columns:')
pass_cols = sorted([c for c in df.columns if 'pass' in c.lower()])
for c in pass_cols:
    vals = df[df['type']=='Pass'][c].dropna()
    if len(vals) > 0:
        sample = vals.iloc[0]
        if isinstance(sample, (list, tuple, np.ndarray)):
            print(f'  {c}: (list) len={len(vals)}, e.g. {sample[:3]}')
        else:
            unique_vals = vals.unique()[:5]
            print(f'  {c}: {unique_vals}')

print('\n' + '=' * 60)
print('POSSESSION & PASSING STATS')
print('=' * 60)

for team in teams:
    team_events = df[df['team'] == team]
    passes = team_events[team_events['type'] == 'Pass'].copy()
    total = len(passes)
    completed = passes[passes['pass_outcome'].isna()]
    acc = len(completed) / total * 100 if total > 0 else 0
    
    print(f'\n{team}')
    print(f'  Passes: {total} | Completed: {len(completed)} | Accuracy: {acc:.1f}%')
    
    if 'pass_length' in passes.columns:
        for length_type in ['Short', 'Long']:
            subset = passes[passes['pass_length'] == length_type]
            sub_comp = subset[subset['pass_outcome'].isna()]
            sub_acc = len(sub_comp) / len(subset) * 100 if len(subset) > 0 else 0
            print(f'    {length_type} passes: {len(subset)} (acc: {sub_acc:.1f}%)')
    
    if 'pass_type' in passes.columns:
        for ptype in ['Cross', 'Through Ball', 'Corner', 'Free Kick', 'Goal Kick', 'Throw-in']:
            subset = passes[passes['pass_type'] == ptype]
            if len(subset) > 0:
                sub_comp = subset[subset['pass_outcome'].isna()]
                sub_acc = len(sub_comp) / len(subset) * 100 if len(subset) > 0 else 0
                print(f'    {ptype}: {len(subset)} (acc: {sub_acc:.1f}%)')
    
    if 'pass_end_location' in completed.columns:
        end_x = completed['pass_end_location'].apply(
            lambda x: x[0] if isinstance(x, (list, tuple, np.ndarray)) and len(x) >= 2 else 0
        )
        final_third = (end_x > 80).sum()
        print(f'    Passes into final third: {final_third}')

In [ ]:
# Shot Map
pitch = Pitch(pitch_type='statsbomb', line_zorder=2, pitch_color='#22313b', line_color='#c7d5cc')
fig, ax = pitch.draw(figsize=(12, 8))

team_colors = {'Argentina': '#75aadb', 'France': '#e55e5e'}

for team in teams:
    shots = df[(df['type'] == 'Shot') & (df['team'] == team)].copy()
    color = team_colors.get(team, 'gray')
    
    for _, shot in shots.iterrows():
        x = shot['location'][0] if isinstance(shot['location'], (list, tuple, np.ndarray)) else 0
        y = shot['location'][1] if isinstance(shot['location'], (list, tuple, np.ndarray)) else 0
        outcome = shot.get('shot_outcome', '')
        
        xg = shot.get('shot_statsbomb_xg', 0)
        if pd.isna(xg):
            xg = 0
        size = max(100, xg * 800)
        
        if outcome == 'Goal':
            marker = '*'
            edgecolor = 'gold'
            linewidth = 2
        elif outcome in ('Saved', 'Saved to Post'):
            marker = 'o'
            edgecolor = 'white'
            linewidth = 1
        elif outcome == 'Blocked':
            marker = 's'
            edgecolor = 'white'
            linewidth = 1
        else:
            marker = '^'
            edgecolor = 'white'
            linewidth = 1
        
        ax.scatter(x, y, s=size, c=color, marker=marker,
                  edgecolors=edgecolor, linewidth=linewidth, alpha=0.85,
                  zorder=5 if outcome == 'Goal' else 3)

# Legend
legend_elements = [
    mpatches.Patch(color='#75aadb', label='Argentina'),
    mpatches.Patch(color='#e55e5e', label='France'),
    plt.scatter([], [], s=100, c='gray', marker='o', edgecolors='white', label='Saved'),
    plt.scatter([], [], s=100, c='gray', marker='*', edgecolors='gold', linewidth=2, label='Goal'),
    plt.scatter([], [], s=100, c='gray', marker='s', edgecolors='white', label='Blocked'),
    plt.scatter([], [], s=100, c='gray', marker='^', edgecolors='white', label='Wide/Off Target'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=10, framealpha=0.9)

plt.title('Shot Map - Argentina vs France (2022 World Cup Final)', fontsize=14, fontweight='bold', color='white',
          loc='center', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Shot Details Table
shots_df = df[df['type'] == 'Shot'].copy()

def get_loc(loc):
    if isinstance(loc, (list, tuple, np.ndarray)) and len(loc) >= 2:
        return f'({loc[0]:.0f}, {loc[1]:.0f})'
    return 'N/A'

shots_df['Location'] = shots_df['location'].apply(get_loc)
shots_df['Player'] = shots_df['player']
shots_df['Team'] = shots_df['team']
shots_df['Outcome'] = shots_df['shot_outcome']
shots_df['xG'] = shots_df['shot_statsbomb_xg'].round(3)
shots_df['Body Part'] = shots_df['shot_body_part']
shots_df['Situation'] = shots_df['shot_type']
shots_df['Minute'] = shots_df['minute']

display_cols = ['Minute', 'Team', 'Player', 'Outcome', 'xG', 'Body Part', 'Situation', 'Location']
print('===== SHOT DETAILS =====')
shots_df[display_cols].sort_values('Minute').reset_index(drop=True)

In [ ]:
# Player Performance Analysis
print('===== PLAYER PERFORMANCE =====')

all_players = df['player'].dropna().unique()
player_stats = []

for player_name in all_players:
    p_events = df[df['player'] == player_name]
    if len(p_events) == 0:
        continue
    team = p_events['team'].iloc[0]
    
    passes = p_events[p_events['type'] == 'Pass']
    total_passes = len(passes)
    completed_passes = len(passes[passes['pass_outcome'].isna()])
    pass_acc = completed_passes / total_passes * 100 if total_passes > 0 else 0
    
    key_passes = len(passes[passes['pass_shot_assist'] == True]) if 'pass_shot_assist' in passes.columns else 0
    assists = len(passes[passes['pass_goal_assist'] == True]) if 'pass_goal_assist' in passes.columns else 0
    
    shots = p_events[p_events['type'] == 'Shot']
    goals = len(shots[shots['shot_outcome'] == 'Goal'])
    xg_sum = shots['shot_statsbomb_xg'].sum()
    
    tackles = len(p_events[p_events['type'] == 'Duel'])
    interceptions = len(p_events[(p_events['type'] == 'Interception') | (p_events['type'] == 'Ball Recovery')])
    fouls = len(p_events[p_events['type'] == 'Foul Committed'])
    fouls_won = len(p_events[p_events['type'] == 'Foul Won'])
    dribbles = len(p_events[p_events['type'] == 'Dribble'])
    
    total_touches = len(p_events)
    
    cards = p_events[p_events['type'] == 'Card']
    yellow = len(cards[(cards['bad_behaviour_card'] == 'Yellow Card') | (cards['foul_committed_card'] == 'Yellow Card')])
    red = len(cards[(cards['bad_behaviour_card'] == 'Red Card') | (cards['foul_committed_card'] == 'Red Card')])
    
    player_stats.append({
        'Player': player_name,
        'Team': team,
        'Touches': total_touches,
        'Passes': total_passes,
        'Pass Acc': round(pass_acc, 1),
        'Key Passes': key_passes,
        'Assists': assists,
        'Shots': len(shots),
        'Goals': goals,
        'xG': round(xg_sum, 3),
        'Tackles': tackles,
        'Interceptions': interceptions,
        'Dribbles': dribbles,
        'Fouls': fouls,
        'Fouls Won': fouls_won,
        'Yellow': yellow,
        'Red': red,
    })

player_df = pd.DataFrame(player_stats)

for team in teams:
    print(f'\n--- {team} - Top Performers ---')
    team_players = player_df[player_df['Team'] == team].sort_values('Touches', ascending=False)
    print(team_players.to_string(index=False))

In [ ]:
# Pass Network
pitch = Pitch(pitch_type='statsbomb', pitch_color='#22313b', line_color='#c7d5cc')

fig, axes = pitch.grid(nrows=1, ncols=2, grid_height=0.8,
                      title_height=0.06, axis=False,
                      title_space=0.02, endnote_space=0.02,
                      figheight=10)

team_colors = {'Argentina': '#75aadb', 'France': '#e55e5e'}

for idx, team in enumerate(teams):
    ax = axes['pitch'][idx]
    
    team_passes = df[(df['team'] == team) & (df['type'] == 'Pass')].copy()
    
    player_positions = []
    for player in team_passes['player'].unique():
        p_passes = team_passes[team_passes['player'] == player]
        if len(p_passes) < 10:
            continue
        p_events = df[(df['player'] == player) & (df['team'] == team)]
        locs = p_events['location'].dropna()
        if len(locs) == 0:
            continue
        avg_x = np.mean([l[0] for l in locs if isinstance(l, (list, tuple, np.ndarray)) and len(l) >= 2])
        avg_y = np.mean([l[1] for l in locs if isinstance(l, (list, tuple, np.ndarray)) and len(l) >= 2])
        player_positions.append({'player': player, 'x': avg_x, 'y': avg_y, 'passes': len(p_passes)})
    
    pos_df = pd.DataFrame(player_positions)
    if len(pos_df) == 0:
        continue
    
    pass_combs = []
    completed_passes = team_passes[team_passes['pass_outcome'].isna()]
    for _, p in completed_passes.iterrows():
        passer = p['player']
        recipient = p.get('pass_recipient', None)
        if pd.isna(recipient):
            continue
        pass_combs.append(tuple(sorted([passer, recipient])))
    
    if len(pass_combs) == 0:
        continue
    
    comb_counts = pd.Series(pass_combs).value_counts().head(10)
    
    color = team_colors.get(team, 'gray')
    size_factor = 500 / pos_df['passes'].max() if pos_df['passes'].max() > 0 else 1
    ax.scatter(pos_df['x'], pos_df['y'],
              s=pos_df['passes'] * size_factor + 100,
              c=color, edgecolors='white', linewidth=1.5, alpha=0.8, zorder=5)
    
    for _, row in pos_df.iterrows():
        if row['passes'] > pos_df['passes'].quantile(0.6):
            name_parts = row['player'].split()
            short_name = name_parts[-1] if len(name_parts) > 1 else name_parts[0]
            ax.text(row['x'], row['y'] - 2.5, short_name, fontsize=7,
                   ha='center', va='top', color='white', fontweight='bold')
    
    for (p1, p2), count in comb_counts.items():
        r1 = pos_df[pos_df['player'] == p1]
        r2 = pos_df[pos_df['player'] == p2]
        if len(r1) == 0 or len(r2) == 0:
            continue
        ax.plot([r1['x'].iloc[0], r2['x'].iloc[0]],
               [r1['y'].iloc[0], r2['y'].iloc[0]],
               color='white', alpha=min(count / comb_counts.max(), 0.8),
               linewidth=min(count / comb_counts.max() * 3, 3), zorder=2)
    
    ax.set_title(f'{team} - Pass Network', fontsize=12, fontweight='bold',
                color='white', pad=10)

plt.tight_layout()
plt.show()

In [ ]:
# Player Heatmaps (Messi & Mbappe)
key_players = ['Lionel Andrés Messi Cuccittini', 'Kylian Mbappé']

pitch = Pitch(pitch_type='statsbomb', pitch_color='#22313b', line_color='#c7d5cc')
fig, axes = pitch.grid(nrows=1, ncols=2, grid_height=0.8,
                      title_height=0.06, axis=False,
                      title_space=0.02, endnote_space=0.02,
                      figheight=8)

for idx, player in enumerate(key_players):
    ax = axes['pitch'][idx]
    
    player_events = df[df['player'] == player].dropna(subset=['location'])
    if len(player_events) == 0:
        ax.set_title(f'{player} - No data', fontsize=10, color='white')
        continue
    
    locs = np.array([loc for loc in player_events['location'] 
                     if isinstance(loc, (list, tuple, np.ndarray)) and len(loc) >= 2])
    
    if len(locs) == 0:
        continue
    
    bin_hex = ax.hexbin(locs[:, 0], locs[:, 1], gridsize=15, cmap='hot',
                       alpha=0.7, mincnt=1, edgecolors='white', linewidths=0.3)
    
    short_name = player.split()[-1]
    ax.set_title(f'{short_name} - Touch Map', fontsize=12, fontweight='bold',
                color='white', pad=10)

plt.tight_layout()
plt.show()

In [ ]:
# Goal Timeline
goals_df = df[(df['type'] == 'Shot') & (df['shot_outcome'] == 'Goal')].copy()

print('===== GOAL TIMELINE =====')
print(f'{"Min":>4} {"Team":<12} {"Scorer":<30} {"Assist":<30} {"xG":<6} {"Body":<12} {"Situation"}')
print('-' * 100)

for _, g in goals_df.sort_values('minute').iterrows():
    minute = g['minute']
    team = g['team']
    player = g['player']
    xg = g.get('shot_statsbomb_xg', 'N/A')
    if pd.notna(xg):
        xg = round(xg, 2)
    body_part = g.get('shot_body_part', 'N/A')
    situation = g.get('shot_type', 'N/A')
    
    # Try to find assist
    assist_info = 'N/A'
    before_events = df[(~df['type'].isin(['Half Start', 'Starting XI'])) &
                       (df['team'] == team) &
                       (df['minute'] < g['minute'] + 2) &
                       (df['minute'] > g['minute'] - 2) &
                       (df['type'] == 'Pass')]
    assist_passes = before_events[before_events['pass_shot_assist'] == True] if 'pass_shot_assist' in before_events.columns else pd.DataFrame()
    if len(assist_passes) > 0:
        assist_info = assist_passes.iloc[-1]['player']
    
    print(f'{minute:>4} {team:<12} {player:<30} {str(assist_info):<30} {str(xg):<6} {body_part:<12} {situation}')

In [ ]:
# xG Comparison Chart
team_xg = []
for team in teams:
    shots = df[(df['type'] == 'Shot') & (df['team'] == team)]
    total_xg = shots['shot_statsbomb_xg'].sum()
    n_goals = len(shots[shots['shot_outcome'] == 'Goal'])
    n_shots = len(shots)
    team_xg.append({'Team': team, 'Shots': n_shots, 'Total xG': round(total_xg, 2), 'Goals': n_goals})

xg_df = pd.DataFrame(team_xg)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(xg_df))
width = 0.25

bars1 = ax.bar(x - width, xg_df['Total xG'], width, label='xG', 
               color=['#75aadb', '#e55e5e'], alpha=0.65, edgecolor='white')
bars2 = ax.bar(x, xg_df['Goals'], width, label='Goals', 
               color=['#75aadb', '#e55e5e'], alpha=1.0, edgecolor='white')
bars3 = ax.bar(x + width, xg_df['Shots'], width, label='Shots', 
               color=['#75aadb', '#e55e5e'], alpha=0.35, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(xg_df['Team'])
ax.set_ylabel('Count / xG')
ax.set_title('xG vs Goals vs Shots', fontweight='bold', fontsize=13)
ax.legend()

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.annotate(f'{height}',
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 3), textcoords='offset points',
                       ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Passing Sonar - Top Passers
for team in teams:
    team_passes = df[(df['team'] == team) & (df['type'] == 'Pass')]
    player_pass_counts = team_passes['player'].value_counts().head(6)
    
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = team_colors.get(team, 'steelblue')
    bars = ax.barh(range(len(player_pass_counts)), player_pass_counts.values,
                  color=colors, alpha=0.8, edgecolor='white', linewidth=1.2)
    
    for i, (player, total) in enumerate(player_pass_counts.items()):
        player_passes = team_passes[team_passes['player'] == player]
        completed = len(player_passes[player_passes['pass_outcome'].isna()])
        acc = completed / total * 100 if total > 0 else 0
        ax.text(total + 1, i, f'{acc:.0f}%', va='center', fontsize=9, fontweight='bold')
    
    ax.set_yticks(range(len(player_pass_counts)))
    ax.set_yticklabels([p.split()[-1] for p in player_pass_counts.index])
    ax.set_xlabel('Number of Passes')
    ax.set_title(f'{team} - Top Passers (with accuracy %)', fontweight='bold', fontsize=12)
    ax.invert_yaxis()
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Defensive Actions Map
pitch = Pitch(pitch_type='statsbomb', pitch_color='#22313b', line_color='#c7d5cc')
fig, axes = pitch.grid(nrows=1, ncols=2, grid_height=0.8,
                      title_height=0.06, axis=False,
                      title_space=0.02, endnote_space=0.02,
                      figheight=10)

for idx, team in enumerate(teams):
    ax = axes['pitch'][idx]
    color = team_colors.get(team, 'gray')
    
    defensive_types = ['Duel', 'Interception', 'Clearance', 'Block']
    defensive = df[(df['team'] == team) & (df['type'].isin(defensive_types))]
    
    for _, ev in defensive.iterrows():
        loc = ev.get('location')
        if not isinstance(loc, (list, tuple, np.ndarray)) or len(loc) < 2:
            continue
        ev_type = ev['type']
        markers = {'Duel': 'o', 'Interception': 'D', 'Clearance': 's', 'Block': 'X'}
        sizes = {'Duel': 40, 'Interception': 50, 'Clearance': 30, 'Block': 60}
        marker = markers.get(ev_type, 'o')
        size = sizes.get(ev_type, 40)
        ax.scatter(loc[0], loc[1], s=size, c=color, marker=marker,
                  edgecolors='white', linewidth=0.5, alpha=0.6)
    
    ax.set_title(f'{team} - Defensive Actions', fontsize=12, fontweight='bold',
                color='white', pad=10)

# Legend
legend_ax = axes['endnote']
legend_elements = [
    plt.scatter([], [], s=40, c='gray', marker='o', edgecolors='white', label='Tackle (Duel)'),
    plt.scatter([], [], s=50, c='gray', marker='D', edgecolors='white', label='Interception'),
    plt.scatter([], [], s=30, c='gray', marker='s', edgecolors='white', label='Clearance'),
    plt.scatter([], [], s=60, c='gray', marker='X', edgecolors='white', label='Block'),
]
legend_ax.legend(handles=legend_elements, loc='center', fontsize=9, ncol=4, frameon=False)
legend_ax.axis('off')

plt.show()

In [ ]:
# Match Event Timeline
fig, ax = plt.subplots(figsize=(14, 6))

significant_events = []

# Goals
for _, g in goals_df.iterrows():
    significant_events.append({
        'minute': g['minute'], 'team': g['team'], 'event': 'Goal',
        'player': g['player'], 'size': 250, 'marker': '*'
    })

# Yellow cards
yellow_cards = df[(df['type'] == 'Card') & ((df['bad_behaviour_card'] == 'Yellow Card') | (df['foul_committed_card'] == 'Yellow Card'))]
for _, c in yellow_cards.iterrows():
    significant_events.append({
        'minute': c['minute'], 'team': c['team'], 'event': 'Yellow Card',
        'player': c['player'], 'size': 100, 'marker': 's'
    })

# Substitutions
subs = df[df['type'] == 'Substitution']
for _, s in subs.iterrows():
    out = s.get('substitution_replacement', s.get('player_out', ''))
    significant_events.append({
        'minute': s['minute'], 'team': s['team'], 'event': 'Sub',
        'player': f"{s['player']} -> {out}", 'size': 80, 'marker': 'o'
    })

events_df = pd.DataFrame(significant_events)

if len(events_df) > 0:
    for _, ev in events_df.iterrows():
        y_pos = 1 if ev['team'] == 'Argentina' else -1
        color = '#75aadb' if ev['team'] == 'Argentina' else '#e55e5e'
        
        ax.scatter(ev['minute'], y_pos, s=ev['size'], c=color,
                  marker=ev['marker'], edgecolors='white', linewidth=1.0,
                  zorder=5, alpha=0.85)
        
        label = f"{ev['minute']}' - {ev['event']}"
        if ev['event'] == 'Goal':
            label += f" - {ev['player'].split()[-1]}"
        offset = 0.15 if y_pos > 0 else -0.15
        ax.text(ev['minute'], y_pos + offset * 2.5, label,
               ha='center', va='bottom' if y_pos > 0 else 'top',
               fontsize=7, color=color, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.6))

ax.axhline(y=0, color='gray', linestyle='-', linewidth=2, alpha=0.5)
ax.set_ylim(-2.5, 2.5)
ax.set_xlim(-2, 125)
ax.set_yticks([1, -1])
ax.set_yticklabels(['Argentina', 'France'], fontsize=11, fontweight='bold')
ax.set_xlabel('Minute', fontsize=11)
ax.set_title('Match Event Timeline - 2022 World Cup Final', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.grid(axis='y', visible=False)

# Half-time, FT, AET markers
ax.axvline(x=45, color='white', linestyle='--', alpha=0.5, linewidth=1)
ax.text(45, 2.1, 'HT', ha='center', fontsize=9, color='white', alpha=0.6)
ax.axvline(x=90, color='white', linestyle='--', alpha=0.5, linewidth=1)
ax.text(90, 2.1, 'FT', ha='center', fontsize=9, color='white', alpha=0.6)
ax.axvline(x=120, color='white', linestyle=':', alpha=0.5, linewidth=1)
ax.text(120, 2.1, 'AET', ha='center', fontsize=9, color='white', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Formation & Substitutions
print('===== FORMATION & SUBSTITUTIONS =====')

for team_name, lineup_df in lineups.items():
    print(f'\n{team_name}:')
    if 'formation' in lineup_df.columns:
        print(f'  Formation: {lineup_df["formation"].iloc[0]}')
    
    # Starting XI
    starters = lineup_df[lineup_df['player_in_starting_lineup'] == True] if 'player_in_starting_lineup' in lineup_df.columns else lineup_df.head(11)
    if 'position_name' in lineup_df.columns and 'player_name' in lineup_df.columns:
        for pos in ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']:
            players_in_pos = lineup_df[lineup_df['position_name'].str.contains(pos, case=False, na=False)]
            if len(players_in_pos) > 0:
                names = players_in_pos['player_name'].tolist()
                print(f'  {pos}s: {", ".join(names)}')
    
    # Substitutions
    subs = df[(df['team'] == team_name) & (df['type'] == 'Substitution')]
    print(f'  Substitutions ({len(subs)}):')
    for _, s in subs.iterrows():
        minute = s['minute']
        player_off = s.get('substitution_replacement', s.get('player_out', 'N/A'))
        player_on = s['player']
        print(f"    {minute}' - {player_off} -> {player_on}")

In [ ]:
# Duels & Ball Recoveries
print('===== DUELS & BALL RECOVERIES =====')

for team in teams:
    team_events = df[df['team'] == team]
    
    duels = team_events[team_events['type'] == 'Duel']
    if len(duels) > 0 and 'duel_outcome' in duels.columns:
        won = duels[duels['duel_outcome'].isin(['Won', 'Success'])]
        print(f'{team}: Duels won {len(won)}/{len(duels)} ({len(won)/len(duels)*100:.1f}%)')
    else:
        print(f'{team}: Duels - {len(duels)} total')
    
    recoveries = team_events[team_events['type'] == 'Ball Recovery']
    print(f'{team}: Ball recoveries - {len(recoveries)}')
    
print()
# Ball possession estimate from pass counts
total_passes = sum(stats[t]['Passes'] for t in teams)
for team in teams:
    poss_pct = stats[team]['Passes'] / total_passes * 100
    print(f'{team}: Estimated possession (by passes) - {poss_pct:.1f}%')

In [ ]:
# Final Summary
print('=' * 60)
print('MATCH SUMMARY REPORT')
print('=' * 60)

match_info = matches[matches['match_id'] == MATCH_ID].iloc[0]
home = match_info['home_team']
away = match_info['away_team']
home_score = match_info['home_score']
away_score = match_info['away_score']

print(f"{home} {home_score} - {away_score} {away} (AET, Argentina won 4-2 on penalties)")
print(f"Competition: FIFA World Cup 2022 - Final")
print(f"Date: 2022-12-18 | Stadium: {match_info['stadium']}")
print()

print('KEY STATISTICS:')
for team in teams:
    s = stats[team]
    print(f"  {team}:")
    print(f"    Possession (by passes): {s['Passes']} passes")
    print(f"    Shots: {s['Shots']} | On Target: {s['Shots on Target']} | Goals: {s['Goals']}")
    print(f"    Fouls: {s['Fouls']} | Cards: {s['Yellow']}Y {s['Red']}R")
    print(f"    Tackles: {s['Tackles']} | Interceptions: {s['Interceptions']}")

print()
print('GOAL SCORERS:')
for _, g in goals_df.sort_values('minute').iterrows():
    xg_val = g.get('shot_statsbomb_xg', 'N/A')
    print(f"  {g['minute']}' - {g['player']} ({g['team']}) xG: {xg_val}")

print()
print('KEY INSIGHTS:')
print(f"  1. Argentina and France played out a 3-3 thriller, with Argentina")
print(f"     prevailing on penalties in one of the greatest World Cup finals.")
print(f"  2. Argentina took {stats[home]['Shots']} shots ({stats[home]['Shots on Target']} on target)")
print(f"     while France had {stats[away]['Shots']} shots ({stats[away]['Shots on Target']} on target).")
print(f"  3. Both teams showed incredible resilience - Argentina dominated early,")
print(f"     France mounted a dramatic second-half comeback.")
print(f"  4. Messi (2 goals) and Mbappe (hat-trick) both delivered")
print(f"     legendary performances on the biggest stage.")

print()
print('=' * 60)
print('END OF REPORT')
print('=' * 60)